# Hybrid setup — Foundry to SQL Managed Instance over a private pathRun each cell in order. **Every cell is safe to re-run** — they check before they create, so ifsomething fails you can fix it and run the notebook again from the top without duplicatingresources.### What this builds| Component | Where it ends up ||---|---|| Foundry project, model, agent | **Public** — unchanged, authenticates to MCP with an Entra token || Container Apps environment | **Inside your VNet**, new delegated subnet || Container registry | **Premium**, private endpoint, public access off || SQL Managed Instance | **Existing** — reached on its VNet-local endpoint, port 1433 |### What it does not doIt does not create SQL MI, and it does not touch the `ManagedInstance` subnet. That subnet isdelegated to `Microsoft.Sql/managedInstances` and cannot host anything else — this notebook adds**new** subnets to the same virtual network instead.### Before you start- **VS Code** with the *Polyglot Notebooks* extension, and the **PowerShell** kernel selected  (top right of this notebook). This will not run in Azure Cloud Shell.- **Azure CLI** signed in.- Permission to create subnets, private endpoints and private DNS zones on the virtual network.The companion explanation for every step is in[`docs/demo-portal-runbook.md`](../docs/demo-portal-runbook.md) under *Private topology variant*.

---## Phase 0 — Configure and preflight

### 0.1 — Set your values**This is the only cell you need to edit.** Everything below reads from it.`$rg` and `$vnetRg` are frequently different — the virtual network hosting SQL MI often lives in anetwork resource group rather than the application one. Setting both to the same value when theydiffer is the most common cause of `ResourceNotFound` later.

In [ ]:
# ---- Application resources -------------------------------------------------$subscriptionId = '<subscription-id>'$rg             = '<app-resource-group>'$location       = '<region>'                    # must match the VNet region$acrName        = '<new-premium-registry-name>' # lowercase alphanumeric, globally unique$envName        = '<new-container-apps-env>'$appName        = 'app-sql-mcp'$identityName   = '<existing-mcp-uami-name>'# ---- Network resources -----------------------------------------------------$vnetRg         = '<vnet-resource-group>'       # RG containing the SQL MI virtual network$vnetName       = '<vnet-hosting-sql-mi>'$acaSubnet      = 'snet-aca-mcp'$peSubnet       = 'snet-private-endpoints'$acaPrefix      = '<e.g. 10.0.8.0/27>'          # /27 or larger, must not overlap$pePrefix       = '<e.g. 10.0.8.32/28>'         # /28 is plenty# ---- SQL Managed Instance (existing) ---------------------------------------$miName         = '<existing-sql-mi-name>'$databaseName   = '<database-the-agent-will-read>'# ---- Foundry / Entra (existing) --------------------------------------------$foundryAccount = '<foundry-account-name>'$foundryProject = '<foundry-project-name>'$mcpAppId       = '<mcp-entra-app-client-id>'   # bare GUID, no api:// prefix$tenantId       = '<tenant-id>'Write-Host 'Values set.' -ForegroundColor Green

### 0.2 — Sign in and select the subscriptionSkip the `az login` line if you are already signed in.

In [ ]:
$ErrorActionPreference = 'Stop'# az login --tenant $tenantId          # uncomment if not already signed inaz account set --subscription $subscriptionId$acct = az account show -o json | ConvertFrom-JsonWrite-Host ("Subscription : {0}" -f $acct.name)Write-Host ("Tenant       : {0}" -f $acct.tenantId)Write-Host ("Signed in as : {0}" -f $acct.user.name)if ($acct.tenantId -ne $tenantId) {    Write-Warning "Signed-in tenant does not match `$tenantId. Check before continuing."}

### 0.3 — Preflight**This cell creates nothing.** It checks every assumption the rest of the notebook depends on, soblockers surface now rather than halfway through with resources half-built.Read the output before continuing. `FAIL` must be resolved. `WARN` is usually informational —notably, resources that already exist are reported so you know what will be reused.

In [ ]:
$results = [System.Collections.Generic.List[object]]::new()function Check($name, $ok, $detail) {    $results.Add([pscustomobject]@{        Check  = $name        Status = if ($ok -eq $null) { 'WARN' } elseif ($ok) { 'PASS' } else { 'FAIL' }        Detail = $detail    })}# --- Resource groups -------------------------------------------------------$rgOk = (az group exists --name $rg) -eq 'true'Check 'App resource group' $rgOk $rg$vnetRgOk = (az group exists --name $vnetRg) -eq 'true'Check 'VNet resource group' $vnetRgOk $vnetRg# --- Virtual network -------------------------------------------------------$vnet = az network vnet show --resource-group $vnetRg --name $vnetName -o json 2>$null | ConvertFrom-Jsonif ($vnet) {    Check 'Virtual network' $true "$vnetName in $($vnet.location)"    Check 'VNet region matches $location' ($vnet.location -eq $location) "vnet=$($vnet.location) location=$location"    $spaces = $vnet.addressSpace.addressPrefixes -join ', '    Check 'VNet address space' $true $spaces    $used = $vnet.subnets | ForEach-Object { "$($_.name) $($_.addressPrefix)" }    Check 'Existing subnets' $null ($used -join ' | ')    # MI subnet - the one we must NOT reuse    $miSubnet = $vnet.subnets | Where-Object { $_.delegations.serviceName -contains 'Microsoft.Sql/managedInstances' }    if ($miSubnet) {        Check 'SQL MI subnet found (will not be touched)' $true "$($miSubnet.name) $($miSubnet.addressPrefix)"    } else {        Check 'SQL MI subnet' $null 'No subnet delegated to Microsoft.Sql/managedInstances in this VNet'    }    # Do the target subnets already exist?    $existingAca = $vnet.subnets | Where-Object { $_.name -eq $acaSubnet }    $existingPe  = $vnet.subnets | Where-Object { $_.name -eq $peSubnet }    if ($existingAca) { Check 'ACA subnet already exists' $null "$acaSubnet -> will be reused/verified" }    if ($existingPe)  { Check 'PE subnet already exists'  $null "$peSubnet -> will be reused/verified" }} else {    Check 'Virtual network' $false "$vnetName not found in $vnetRg"}# --- SQL Managed Instance --------------------------------------------------$mi = az sql mi show --name $miName --resource-group $vnetRg -o json 2>$null | ConvertFrom-Jsonif (-not $mi) { $mi = az sql mi list -o json 2>$null | ConvertFrom-Json | Where-Object { $_.name -eq $miName } | Select-Object -First 1 }if ($mi) {    Check 'SQL Managed Instance' $true "$($mi.name) in $($mi.location)"    Check 'SQL MI VNet-local FQDN' $true $mi.fullyQualifiedDomainName    Check 'SQL MI public endpoint' $null ("publicDataEndpointEnabled = {0}" -f $mi.publicDataEndpointEnabled)    $miFqdn = $mi.fullyQualifiedDomainName} else {    Check 'SQL Managed Instance' $false "$miName not found"}# --- Managed identity ------------------------------------------------------$uami = az identity show --name $identityName --resource-group $rg -o json 2>$null | ConvertFrom-Jsonif ($uami) {    Check 'MCP managed identity' $true "clientId $($uami.clientId)"} else {    Check 'MCP managed identity' $false "$identityName not found in $rg"}# --- Foundry ---------------------------------------------------------------$foundry = az cognitiveservices account show --name $foundryAccount --resource-group $rg -o json 2>$null | ConvertFrom-JsonCheck 'Foundry account' ($null -ne $foundry) $(if ($foundry) { "$foundryAccount ($($foundry.location))" } else { "$foundryAccount not found" })# --- Registry name availability --------------------------------------------$acrCheck = az acr check-name --name $acrName -o json 2>$null | ConvertFrom-Jsonif ($acrCheck) {    if ($acrCheck.nameAvailable) {        Check 'Registry name available' $true $acrName    } else {        $mine = az acr show --name $acrName -o json 2>$null | ConvertFrom-Json        if ($mine) { Check 'Registry already exists' $null "$acrName sku=$($mine.sku.name) -> will be reused/verified" }        else { Check 'Registry name available' $false "$acrName taken by another subscription: $($acrCheck.message)" }    }}# --- Permission probe (read-only) ------------------------------------------$canWriteNet = $truetry { az network vnet subnet list --resource-group $vnetRg --vnet-name $vnetName -o none 2>$null } catch { $canWriteNet = $false }Check 'Can read VNet subnets' $canWriteNet 'Write permission is not probed - confirm you hold Network Contributor'# --- Report ----------------------------------------------------------------$results | Format-Table -AutoSize -Wrap$fails = @($results | Where-Object Status -eq 'FAIL')if ($fails.Count -gt 0) {    Write-Host ''    Write-Host "$($fails.Count) check(s) FAILED - resolve before continuing." -ForegroundColor Red} else {    Write-Host ''    Write-Host 'Preflight passed. Review any WARN rows, then continue.' -ForegroundColor Green}

> **On Directory Readers.** Creating the MCP database user on SQL MI needs the Entra> **Directory Readers** role on the managed instance's server identity. It cannot be checked from> the CLI here. If step 3.2 fails with a principal-resolution error, that is the cause — it needs a> Privileged Role Administrator or Global Administrator to grant, and usually follows a slower> approval path than Azure RBAC. Start that request early.

---## Phase 1 — NetworkAdds two subnets to the **existing** virtual network. The SQL MI subnet is not modified.

### 1.1 — Container Apps subnet, delegated`/27` or larger for a workload profiles environment. Must be dedicated — nothing else may use it.

In [ ]:
$existing = az network vnet subnet show --resource-group $vnetRg --vnet-name $vnetName `    --name $acaSubnet -o json 2>$null | ConvertFrom-Jsonif (-not $existing) {    Write-Host "Creating subnet $acaSubnet ($acaPrefix)..."    az network vnet subnet create --resource-group $vnetRg --vnet-name $vnetName `        --name $acaSubnet --address-prefixes $acaPrefix -o none} else {    Write-Host "Subnet $acaSubnet already exists ($($existing.addressPrefix)) - skipping create."}# Delegation is idempotent; re-applying is harmlessaz network vnet subnet update --resource-group $vnetRg --vnet-name $vnetName `    --name $acaSubnet --delegations Microsoft.App/environments -o none$check = az network vnet subnet show --resource-group $vnetRg --vnet-name $vnetName `    --name $acaSubnet --query "{prefix:addressPrefix, delegation:delegations[0].serviceName}" -o json | ConvertFrom-JsonWrite-Host ("  prefix     : {0}" -f $check.prefix)Write-Host ("  delegation : {0}" -f $check.delegation) -ForegroundColor $(if ($check.delegation -eq 'Microsoft.App/environments') { 'Green' } else { 'Red' })

### 1.2 — Private endpoint subnet, **no** delegationA delegation here reproduces the *"subnet has a delegation and cannot be used"* error. Privateendpoint network policies must also be `Disabled`.

In [ ]:
$existing = az network vnet subnet show --resource-group $vnetRg --vnet-name $vnetName `    --name $peSubnet -o json 2>$null | ConvertFrom-Jsonif (-not $existing) {    Write-Host "Creating subnet $peSubnet ($pePrefix)..."    az network vnet subnet create --resource-group $vnetRg --vnet-name $vnetName `        --name $peSubnet --address-prefixes $pePrefix -o none} else {    Write-Host "Subnet $peSubnet already exists ($($existing.addressPrefix)) - skipping create."}az network vnet subnet update --resource-group $vnetRg --vnet-name $vnetName `    --name $peSubnet --disable-private-endpoint-network-policies true -o none$check = az network vnet subnet show --resource-group $vnetRg --vnet-name $vnetName `    --name $peSubnet --query "{prefix:addressPrefix, delegation:delegations[0].serviceName, policies:privateEndpointNetworkPolicies}" -o json | ConvertFrom-JsonWrite-Host ("  prefix     : {0}" -f $check.prefix)Write-Host ("  delegation : {0}" -f $(if ($check.delegation) { $check.delegation } else { 'none (correct)' })) -ForegroundColor $(if ($check.delegation) { 'Red' } else { 'Green' })Write-Host ("  policies   : {0}" -f $check.policies) -ForegroundColor $(if ($check.policies -eq 'Disabled') { 'Green' } else { 'Red' })

### 1.3 — Private DNS zone for the registryCreated now, populated with records later when the private endpoint exists.

In [ ]:
$zone = 'privatelink.azurecr.io'$exists = az network private-dns zone show --resource-group $vnetRg --name $zone -o json 2>$null | ConvertFrom-Jsonif (-not $exists) {    Write-Host "Creating private DNS zone $zone..."    az network private-dns zone create --resource-group $vnetRg --name $zone -o none} else {    Write-Host "Zone $zone already exists - skipping create."}$linkName = "link-$vnetName"$link = az network private-dns link vnet show --resource-group $vnetRg --zone-name $zone `    --name $linkName -o json 2>$null | ConvertFrom-Jsonif (-not $link) {    Write-Host "Linking zone to $vnetName..."    az network private-dns link vnet create --resource-group $vnetRg --zone-name $zone `        --name $linkName --virtual-network $vnetName --registration-enabled false -o none} else {    Write-Host "Link $linkName already exists - skipping create."}az network private-dns link vnet list --resource-group $vnetRg --zone-name $zone `    --query "[].{link:name, state:provisioningState}" -o table

### 1.4 — Verify the network, and check DNSIf `dnsServers` comes back empty, the VNet uses Azure-provided DNS and private zones resolveautomatically. If custom DNS servers are listed, **they must forward `privatelink.azurecr.io` toAzure DNS at `168.63.129.16`** or private resolution will fail even though everything here isconfigured correctly. Confirm with whoever runs those servers.

In [ ]:
$aca = az network vnet subnet show --resource-group $vnetRg --vnet-name $vnetName --name $acaSubnet `    --query "{name:name, prefix:addressPrefix, delegation:delegations[0].serviceName}" -o json | ConvertFrom-Json$pe  = az network vnet subnet show --resource-group $vnetRg --vnet-name $vnetName --name $peSubnet `    --query "{name:name, prefix:addressPrefix, delegation:delegations[0].serviceName, policies:privateEndpointNetworkPolicies}" -o json | ConvertFrom-Json$dns = az network vnet show --resource-group $vnetRg --name $vnetName --query dhcpOptions.dnsServers -o json | ConvertFrom-Json$ok = ($aca.delegation -eq 'Microsoft.App/environments') -and (-not $pe.delegation) -and ($pe.policies -eq 'Disabled')[pscustomobject]@{ Subnet=$aca.name; Prefix=$aca.prefix; Delegation=$aca.delegation; Policies='n/a' },[pscustomobject]@{ Subnet=$pe.name;  Prefix=$pe.prefix;  Delegation=$(if($pe.delegation){$pe.delegation}else{'none'}); Policies=$pe.policies } |    Format-Table -AutoSizeif ($dns -and $dns.Count -gt 0) {    Write-Host ("Custom DNS servers: {0}" -f ($dns -join ', ')) -ForegroundColor Yellow    Write-Host 'These must forward privatelink.azurecr.io to 168.63.129.16.' -ForegroundColor Yellow} else {    Write-Host 'Azure-provided DNS - private zones resolve automatically.' -ForegroundColor Green}Write-Host ''if ($ok) { Write-Host 'PHASE 1 PASS' -ForegroundColor Green } else { Write-Host 'PHASE 1 FAIL - check the table above' -ForegroundColor Red }

---## Phase 2 — Container registry**Order matters here.** The image is built and pushed while the registry is still publiclyreachable, and public access is disabled afterwards. `az acr build` stops working once publicaccess is off — ACR Tasks need public IPs unless you assign a dedicated agent pool.

### 2.1 — Create the Premium registry, public access still onPremium is required: private endpoints are a Premium-tier feature. A Basic or Standard registrycannot be upgraded in place for this purpose.

In [ ]:
$acr = az acr show --name $acrName -o json 2>$null | ConvertFrom-Jsonif (-not $acr) {    Write-Host "Creating Premium registry $acrName..."    az acr create --name $acrName --resource-group $rg --location $location --sku Premium -o none    $acr = az acr show --name $acrName -o json | ConvertFrom-Json} else {    Write-Host "Registry $acrName already exists (sku $($acr.sku.name))."    if ($acr.sku.name -ne 'Premium') {        Write-Host 'Upgrading to Premium...' -ForegroundColor Yellow        az acr update --name $acrName --sku Premium -o none    }}# ARM audience tokens are required for managed-identity image pullaz acr config authentication-as-arm update -r $acrName --status enabled -o none$acrId = az acr show --name $acrName --query id -o tsv$loginServer = az acr show --name $acrName --query loginServer -o tsvWrite-Host ("  loginServer : {0}" -f $loginServer)Write-Host ("  sku         : {0}" -f (az acr show --name $acrName --query sku.name -o tsv))Write-Host ("  adminUser   : {0}" -f (az acr show --name $acrName --query adminUserEnabled -o tsv))

### 2.2 — Grant `AcrPull` to the MCP identityThis is what lets the Container App pull the image without registry credentials. Note the earlierportal error *"admin credentials on the ACR are disabled"* is resolved by this plus`--registry-identity` later — **not** by enabling the admin user.

In [ ]:
$identityId   = az identity show --name $identityName --resource-group $rg --query id -o tsv$identityPrin = az identity show --name $identityName --resource-group $rg --query principalId -o tsv$existing = az role assignment list --assignee $identityPrin --scope $acrId `    --role AcrPull -o json 2>$null | ConvertFrom-Jsonif (-not $existing -or $existing.Count -eq 0) {    Write-Host 'Assigning AcrPull...'    az role assignment create --assignee-object-id $identityPrin `        --assignee-principal-type ServicePrincipal --role AcrPull --scope $acrId -o none    Start-Sleep -Seconds 10   # role assignments take a moment to propagate} else {    Write-Host 'AcrPull already assigned - skipping.'}az role assignment list --assignee $identityPrin --scope $acrId `    --query "[].{role:roleDefinitionName, scope:scope}" -o table

### 2.3 — Build and push the imageRun this from the repository root. The build substitutes your MCP application client ID and tenantID into the DAB configuration — a mismatch here surfaces much later as a `401` from the agent, sothe output is checked below.

In [ ]:
# Resolve the repo root by looking for a known marker, not by assuming $PWD$repoRoot = $PWDwhile ($repoRoot -and -not (Test-Path (Join-Path $repoRoot 'azure.yaml'))) {    $parent = Split-Path -Parent $repoRoot    if ($parent -eq $repoRoot) { break }    $repoRoot = $parent}if (-not (Test-Path (Join-Path $repoRoot 'azure.yaml'))) {    throw "Could not locate the repository root. Set `$repoRoot manually."}Write-Host "Repo root: $repoRoot"Push-Location $repoRoottry {    & ./scripts/build-demo-mcp.ps1 -RegistryName $acrName -McpApplicationId $mcpAppId -TenantId $tenantId} finally {    Pop-Location}Write-Host ''Write-Host 'Tags in registry:'az acr repository show-tags --name $acrName --repository sql-mcp -o table

### 2.4 — Private endpoint and DNS recordsConfiguring a private endpoint automatically enables *dedicated data endpoints*, so the zone needsa record for the registry **and** one per region for the data endpoint. The `dns-zone-group`command creates all of them — creating records by hand and missing the data endpoint produces imagepulls that authenticate and then hang.

In [ ]:
$acrId  = az acr show --name $acrName --query id -o tsv$peName = "pe-$acrName"$pe = az network private-endpoint show --name $peName --resource-group $vnetRg -o json 2>$null | ConvertFrom-Jsonif (-not $pe) {    Write-Host "Creating private endpoint $peName..."    az network private-endpoint create `        --name $peName --resource-group $vnetRg `        --vnet-name $vnetName --subnet $peSubnet `        --private-connection-resource-id $acrId `        --group-ids registry --connection-name "conn-$acrName" -o none} else {    Write-Host "Private endpoint $peName already exists - skipping create."}# Resolve the zone by ID so this works across resource groups$zoneId = az network private-dns zone show --resource-group $vnetRg --name 'privatelink.azurecr.io' --query id -o tsv$zg = az network private-endpoint dns-zone-group show --endpoint-name $peName `    --resource-group $vnetRg --name 'default' -o json 2>$null | ConvertFrom-Jsonif (-not $zg) {    Write-Host 'Creating DNS zone group...'    az network private-endpoint dns-zone-group create `        --resource-group $vnetRg --endpoint-name $peName `        --name 'default' --private-dns-zone $zoneId --zone-name 'acr' -o none} else {    Write-Host 'DNS zone group already exists - skipping create.'}Write-Host ''Write-Host 'Records now in the zone (expect the registry AND a .data. record):'az network private-dns record-set a list --resource-group $vnetRg --zone-name 'privatelink.azurecr.io' `    --query "[].{name:name, ip:aRecords[0].ipv4Address}" -o table

### 2.5 — Lock the registry downOnly after the image is pushed. Verify the tag exists in the output above before running this.

In [ ]:
$tags = az acr repository show-tags --name $acrName --repository sql-mcp -o json 2>$null | ConvertFrom-Jsonif (-not $tags -or $tags.Count -eq 0) {    Write-Host 'STOP - no image tags found. Do not disable public access yet.' -ForegroundColor Red    Write-Host 'Re-run cell 2.3 first; az acr build will not work once public access is off.' -ForegroundColor Red} else {    Write-Host ("Found {0} tag(s). Disabling public network access..." -f $tags.Count)    az acr update --name $acrName --public-network-enabled false -o none    Write-Host ("  publicNetworkAccess : {0}" -f (az acr show --name $acrName --query publicNetworkAccess -o tsv)) -ForegroundColor Green}

---## Phase 3 — SQL contract on the managed instanceThe agent reads only curated views, granted to a dedicated `mcp_reader` role. SQL is the finalauthority — no configuration elsewhere can grant access the database denies.

### 3.1 — Generate the SQLThis produces the script to run against your managed instance. Review it before executing — thegrants define exactly what the agent can see.Run it from somewhere that can reach the instance: your SSMS server, or Cloud Shell if the MIpublic endpoint is currently enabled.

In [ ]:
$uamiName = $identityName$sql = @"-- Run against: $miFqdn , database [$databaseName]-- Creates the least-privilege role and maps the MCP managed identity to it.USE [$databaseName];GOIF DATABASE_PRINCIPAL_ID(N'mcp_reader') IS NULL    CREATE ROLE [mcp_reader] AUTHORIZATION [dbo];GO-- The MCP identity as a contained database user.-- Requires Entra Directory Readers on the SQL MI server identity.IF DATABASE_PRINCIPAL_ID(N'$uamiName') IS NULL    CREATE USER [$uamiName] FROM EXTERNAL PROVIDER;GOIF NOT EXISTS (    SELECT 1 FROM sys.database_role_members rm    JOIN sys.database_principals r ON r.principal_id = rm.role_principal_id    JOIN sys.database_principals m ON m.principal_id = rm.member_principal_id    WHERE r.name = N'mcp_reader' AND m.name = N'$uamiName')    ALTER ROLE [mcp_reader] ADD MEMBER [$uamiName];GO-- ----------------------------------------------------------------------------- GRANT on named objects only. Never db_datareader.-- Replace these with your curated views and procedures.-- ----------------------------------------------------------------------------- GRANT SELECT  ON OBJECT::mcp.vw_example      TO [mcp_reader];-- GRANT EXECUTE ON OBJECT::mcp.usp_example     TO [mcp_reader];GO"@$path = Join-Path $PWD 'mcp-sql-contract.sql'Set-Content -LiteralPath $path -Value $sql -Encoding utf8NoBOMWrite-Host "Written to: $path" -ForegroundColor GreenWrite-Host ''Write-Host $sql

> **If a view reads across databases**, a contained user is not enough — the caller cannot be> resolved in the second database and startup fails with *"The server principal ... is not able to> access the database ..."*. SQL MI supports a server-level login for this:>> ```sql> USE master;> CREATE LOGIN [<uami-name>] FROM EXTERNAL PROVIDER;> -- then in EVERY database the view touches:> DROP USER IF EXISTS [<uami-name>];> CREATE USER [<uami-name>] FROM LOGIN [<uami-name>];> ALTER ROLE [mcp_reader] ADD MEMBER [<uami-name>];> ```>> The `DROP USER` matters — an existing contained user blocks the login-backed one.

### 3.2 — Verify the grantsRun this against the managed instance after applying the contract. Expect only your curated objectsand exactly one role member.

In [ ]:
$verify = @"SELECT s.name AS [schema], o.name AS [object], p.permission_name, p.state_descFROM sys.database_permissions AS pJOIN sys.database_principals AS dp ON dp.principal_id = p.grantee_principal_idJOIN sys.objects AS o ON o.object_id = p.major_idJOIN sys.schemas AS s ON s.schema_id = o.schema_idWHERE dp.name = N'mcp_reader'ORDER BY s.name, o.name;SELECT r.name AS role_name, m.name AS member_nameFROM sys.database_role_members rmJOIN sys.database_principals r ON r.principal_id = rm.role_principal_idJOIN sys.database_principals m ON m.principal_id = rm.member_principal_idWHERE r.name = N'mcp_reader';"@$path = Join-Path $PWD 'mcp-sql-verify.sql'Set-Content -LiteralPath $path -Value $verify -Encoding utf8NoBOMWrite-Host "Written to: $path" -ForegroundColor GreenWrite-Host $verify

---## Phase 4 — Container AppsThe environment is injected into the subnet from Phase 1. Ingress stays **external** — Foundry ispublic and reaches the MCP endpoint over the internet, authenticated by an Entra token. The networkis not the trust boundary on that hop.

### 4.1 — VNet-injected environmentThis takes a few minutes.

In [ ]:
$acaEnv = az containerapp env show --name $envName --resource-group $rg -o json 2>$null | ConvertFrom-Jsonif (-not $acaEnv) {    $acaSubnetId = az network vnet subnet show --resource-group $vnetRg `        --vnet-name $vnetName --name $acaSubnet --query id -o tsv    Write-Host "Creating VNet-injected environment $envName (this takes several minutes)..."    az containerapp env create --name $envName --resource-group $rg --location $location `        --infrastructure-subnet-resource-id $acaSubnetId --enable-workload-profiles -o none    $acaEnv = az containerapp env show --name $envName --resource-group $rg -o json | ConvertFrom-Json} else {    Write-Host "Environment $envName already exists - skipping create."}[pscustomobject]@{    Name          = $acaEnv.name    State         = $acaEnv.properties.provisioningState    Subnet        = ($acaEnv.properties.vnetConfiguration.infrastructureSubnetId -split '/')[-1]    Internal      = $acaEnv.properties.vnetConfiguration.internal    DefaultDomain = $acaEnv.properties.defaultDomain} | Format-List

### 4.2 — Container AppRegistry authentication uses the managed identity, not credentials. The connection string uses the**VNet-local** SQL MI FQDN on port **1433** — not the `.public.` FQDN, and not 3342.

In [ ]:
# Re-resolve rather than depend on earlier cells, so this can run standalone$loginServer = az acr show --name $acrName --query loginServer -o tsv$identityId  = az identity show --name $identityName --resource-group $rg --query id -o tsv$uamiClient  = az identity show --name $identityName --resource-group $rg --query clientId -o tsvif (-not $miFqdn) {    $miFqdn = (az sql mi list -o json | ConvertFrom-Json | Where-Object { $_.name -eq $miName } | Select-Object -First 1).fullyQualifiedDomainName}if (-not $miFqdn) { throw 'SQL MI FQDN could not be resolved. Re-run the preflight cell.' }$imageTag = az acr repository show-tags --name $acrName --repository sql-mcp --top 1 --orderby time_desc -o tsvif (-not $imageTag) { throw 'No image tag found in the registry. Run cell 2.3 first.' }$image    = "$loginServer/sql-mcp:$imageTag"$connStr  = "Server=tcp:$miFqdn,1433;Initial Catalog=$databaseName;Authentication=Active Directory Managed Identity;User Id=$uamiClient;Encrypt=True;TrustServerCertificate=False;Connection Timeout=30;"Write-Host "Image : $image"Write-Host "SQL   : $miFqdn,1433 / $databaseName"Write-Host ''$app = az containerapp show --name $appName --resource-group $rg -o json 2>$null | ConvertFrom-Jsonif (-not $app) {    Write-Host "Creating container app $appName..."    az containerapp create --name $appName --resource-group $rg --environment $envName `        --user-assigned $identityId --registry-identity $identityId `        --registry-server $loginServer --image $image `        --target-port 5000 --ingress external --transport http `        --cpu 0.5 --memory 1.0Gi --min-replicas 1 --max-replicas 1 `        --env-vars "DAB_ENVIRONMENT=Production" "DATABASE_CONNECTION_STRING=$connStr" -o none} else {    Write-Host "Container app exists - updating to image $imageTag..."    az containerapp update --name $appName --resource-group $rg --image $image `        --set-env-vars "DAB_ENVIRONMENT=Production" "DATABASE_CONNECTION_STRING=$connStr" -o none}$app = az containerapp show --name $appName --resource-group $rg -o json | ConvertFrom-Json$mcpFqdn = $app.properties.configuration.ingress.fqdnWrite-Host ("  FQDN  : https://{0}/mcp" -f $mcpFqdn) -ForegroundColor Green

### 4.3 — Verify the revision is healthyIf the revision is unhealthy, the log output below identifies which layer failed. The containerresolves every configured entity against the database at startup, so the first thirty seconds arethe most informative moment in the whole process.

In [ ]:
$rev = az containerapp revision list --name $appName --resource-group $rg `    --query "[0].{name:name, active:properties.active, state:properties.runningState, replicas:properties.replicas}" -o json | ConvertFrom-Json$rev | Format-Listif ($rev.state -ne 'Running') {    Write-Host 'Revision not running - last 60 log lines:' -ForegroundColor Yellow    az containerapp logs show --name $appName --resource-group $rg --tail 60    Write-Host ''    Write-Host 'Common causes:' -ForegroundColor Yellow    Write-Host '  "is not able to access the database"  -> cross-database view, needs a server-level login (3.1 note)'    Write-Host '  "Login failed"                        -> identity not in mcp_reader, or Directory Readers missing'    Write-Host '  "Cannot obtain schema for entity"     -> object missing or not granted'    Write-Host '  connection timeout                    -> DNS or routing to the MI VNet-local FQDN'} else {    Write-Host 'PHASE 4 PASS - revision running' -ForegroundColor Green}

---## Phase 5 — Foundry connection and agentUnchanged from the public topology. The connection stores the MCP endpoint and the audience; theagent references the connection by name.Note the two audience forms are **intentionally different**: the Foundry connection uses`api://<client-id>`, while DAB validates the bare `<client-id>` GUID.

In [ ]:
$audience = "api://$mcpAppId"$endpoint = "https://$mcpFqdn/mcp"Write-Host "Create the project connection in the Foundry portal:"Write-Host "  Project  : $foundryProject"Write-Host "  Name     : sql-mcp"Write-Host "  Endpoint : $endpoint"Write-Host "  Auth     : Microsoft Entra - project managed identity"Write-Host "  Audience : $audience"Write-Host ''Write-Host 'Then confirm the connection JSON shows:'Write-Host '  category: RemoteTool'Write-Host '  authType: ProjectManagedIdentity'Write-Host '  useWorkspaceManagedIdentity: true'

### 5.2 — Register the agentSet the environment variables the registration script expects, then run it.

In [ ]:
$projEndpoint = "https://$foundryAccount.services.ai.azure.com/api/projects/$foundryProject"$env:AZURE_AI_PROJECT_ENDPOINT     = $projEndpoint$env:AZURE_AI_MODEL_DEPLOYMENT_NAME = '<your-model-deployment-name>'$env:MCP_ENDPOINT                   = "https://$mcpFqdn/mcp"$env:MCP_PROJECT_CONNECTION_NAME    = 'sql-mcp'Write-Host "Project endpoint : $projEndpoint"Write-Host "MCP endpoint     : $($env:MCP_ENDPOINT)"Write-Host ''Write-Host 'Set AZURE_AI_MODEL_DEPLOYMENT_NAME above, then run:'Write-Host '  ./scripts/register-demo-agent.ps1' -ForegroundColor CyanWrite-Host ''Write-Host 'Remember to update ALLOWED_TOOLS in src/agent/register_agent.py if you exposed'Write-Host 'your own stored procedures - the three generic tools stay, the custom ones change.'

---## Phase 6 — Close the public path and validateThe last two checks are the ones that matter. Together they prove the data path is private whilethe control path still works — which is the claim this whole topology exists to support, and theevidence a security reviewer will ask for.

### 6.1 — Disable the SQL MI public endpointOnly once the container is healthy and answering. This is the step that makes the topology private.

In [ ]:
$mi = az sql mi show --name $miName --resource-group $vnetRg -o json 2>$null | ConvertFrom-Jsonif (-not $mi) { $mi = az sql mi list -o json | ConvertFrom-Json | Where-Object { $_.name -eq $miName } | Select-Object -First 1 }Write-Host ("Current publicDataEndpointEnabled : {0}" -f $mi.publicDataEndpointEnabled)Write-Host ''Write-Host 'To disable, uncomment and run:' -ForegroundColor CyanWrite-Host "  az sql mi update --name $miName --resource-group $vnetRg --public-data-endpoint-enabled false" -ForegroundColor CyanWrite-Host ''Write-Host 'Check first whether anything else uses the public endpoint - the earlier note said' -ForegroundColor YellowWrite-Host 'it was enabled for an on-premises SSMS server.' -ForegroundColor Yellow# az sql mi update --name $miName --resource-group $vnetRg --public-data-endpoint-enabled false -o none

### 6.2 — Final validation

In [ ]:
$checks = [System.Collections.Generic.List[object]]::new()function V($n, $ok, $d) { $checks.Add([pscustomobject]@{ Check=$n; Status=$(if($ok){'PASS'}else{'FAIL'}); Detail=$d }) }$acrPublic = az acr show --name $acrName --query publicNetworkAccess -o tsvV 'Registry public access disabled' ($acrPublic -eq 'Disabled') "publicNetworkAccess = $acrPublic"$envCfg = az containerapp env show --name $envName --resource-group $rg -o json | ConvertFrom-JsonV 'Environment is VNet-injected' ($null -ne $envCfg.properties.vnetConfiguration.infrastructureSubnetId) `  (($envCfg.properties.vnetConfiguration.infrastructureSubnetId -split '/')[-1])$revState = az containerapp revision list --name $appName --resource-group $rg --query "[0].properties.runningState" -o tsvV 'Container revision running' ($revState -eq 'Running') "state = $revState"$appCfg = az containerapp show --name $appName --resource-group $rg -o json | ConvertFrom-Json$cs = ($appCfg.properties.template.containers[0].env | Where-Object name -eq 'DATABASE_CONNECTION_STRING').valueV 'Using VNet-local SQL endpoint' ($cs -notmatch '\.public\.' -and $cs -match ',1433') 'no .public. infix, port 1433'$miNow = az sql mi list -o json | ConvertFrom-Json | Where-Object { $_.name -eq $miName } | Select-Object -First 1V 'SQL MI public endpoint disabled' (-not $miNow.publicDataEndpointEnabled) "publicDataEndpointEnabled = $($miNow.publicDataEndpointEnabled)"$checks | Format-Table -AutoSize -WrapWrite-Host ''Write-Host 'Two manual checks remain:' -ForegroundColor CyanWrite-Host '  1. Connect to the .public. FQDN on 3342 - this must FAIL.'Write-Host '  2. Ask the agent a question in the Foundry playground - this must WORK.'Write-Host ''Write-Host 'Also prove the denials, which is what a security review will ask for:' -ForegroundColor CyanWrite-Host '  - Ask the agent for data from a base table  -> must be refused'Write-Host '  - Ask the agent to change something         -> must be refused'

---## If you need to start overNothing here is destructive to SQL MI or the virtual network. To remove only what this notebookcreated:```powershellaz containerapp delete --name $appName --resource-group $rg --yesaz containerapp env delete --name $envName --resource-group $rg --yesaz network private-endpoint delete --name "pe-$acrName" --resource-group $vnetRgaz acr delete --name $acrName --resource-group $rg --yesaz network vnet subnet delete --resource-group $vnetRg --vnet-name $vnetName --name $acaSubnetaz network vnet subnet delete --resource-group $vnetRg --vnet-name $vnetName --name $peSubnet```The `ManagedInstance` subnet, the managed instance, and the Foundry project are untouched.